In [8]:
# Cell 1: Install core libraries
!pip install -q transformers datasets torchaudio accelerate jiwer evaluate onnx onnxruntime soundfile librosa pydub

In [9]:
# Cell 2: Build CTC Tokenizer Vocabulary
import json
import os

# Complete Sinhala Unicode character inventory + special tokens
sinhala_vocab = [
    "<pad>", "<s>", "</s>", "<unk>", "|",  # '|' serves as the word/space delimiter
    # Vowels & Modifiers
    "අ", "ආ", "ඇ", "ඈ", "ඉ", "ඊ", "උ", "ඌ", "ඍ", "ඎ", "එ", "ඒ", "ඓ", "ඔ", "ඕ", "ඖ",
    "ං", "ඃ",
    # Consonants & Sanyaka
    "ක", "ඛ", "ග", "ඝ", "ඞ", "ඟ",
    "ච", "ඡ", "ජ", "ඣ", "ඤ", "ඦ",
    "ට", "ඨ", "ඩ", "ඪ", "ණ", "ඬ",
    "ත", "ථ", "ද", "ධ", "න", "ඳ",
    "ප", "ඵ", "බ", "භ", "ම", "ඹ",
    "ය", "ර", "ල", "ව", "ශ", "ෂ", "ස", "හ", "ළ", "ෆ", "ඥ",
    # Diacritics (Pili)
    "්", "ා", "ැ", "ෑ", "ි", "ී", "ු", "ූ", "ෘ", "ෙ", "ේ", "ෛ", "ො", "ෝ", "ෞ"
]

vocab_dict = {char: idx for idx, char in enumerate(sinhala_vocab)}

os.makedirs("./vocab_sinhala", exist_ok=True)
with open("./vocab_sinhala/vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab_dict, f, ensure_ascii=False, indent=2)

print(f"Created vocab.json with {len(vocab_dict)} tokens.")

Created vocab.json with 79 tokens.


In [10]:
# Cell 3: Setup Hugging Face Processor
from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor
)

tokenizer = Wav2Vec2CTCTokenizer(
    "./vocab_sinhala/vocab.json",
    unk_token="<unk>",
    pad_token="<pad>",
    word_delimiter_token="|"
)

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True
)

processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)
processor.save_pretrained("./sinhala_processor")

['./sinhala_processor/processor_config.json']

In [11]:
# Step 1: Download and Extract the OpenSLR 30 Sinhala Dataset
import os

!wget -q --show-progress https://www.openslr.org/resources/30/si_lk.tar.gz
!wget -q --show-progress https://www.openslr.org/resources/30/si_lk.lines.txt

# Extract audio files into a dedicated directory
!mkdir -p ./openslr_sinhala_audio
!tar -xzf si_lk.tar.gz -C ./openslr_sinhala_audio

si_lk.tar.gz.1      100%[===================>] 667.21M  18.1MB/s    in 38s     
si_lk.lines.txt.1   100%[===================>] 193.08K   426KB/s    in 0.5s    


In [12]:
# Step 2: Parse `si_lk.lines.txt` & Build the DataFrame
import re
import os
import glob
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Index all extracted .wav files by filename stem (e.g., "sin_1234")
audio_files = glob.glob("./openslr_sinhala_audio/**/*.wav", recursive=True)
audio_map = {os.path.splitext(os.path.basename(f))[0]: os.path.abspath(f) for f in audio_files}
print(f"Found {len(audio_map)} extracted audio files.")

# 2. Parse `si_lk.lines.txt`
# OpenSLR 30 format per line: ( audio_id "transcript text" ) or audio_id\t"transcript text"
records = []
pattern = re.compile(r'\(?\s*([a-zA-Z0-9_\-]+)\s+["\']?([^"\']+)["\']?\s*\)?')

with open("si_lk.lines.txt", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        match = pattern.match(line)
        if match:
            audio_id, text = match.groups()
            text = text.strip()

            # Clean punctuation, quotes, and normalize whitespace
            text = re.sub(r'[^\u0D80-\u0DFF\s]', '', text)  # Keep only Sinhala Unicode + spaces
            text = re.sub(r'\s+', ' ', text).strip()

            if audio_id in audio_map and len(text) > 0:
                records.append({
                    "audio_path": audio_map[audio_id],
                    "transcript": text
                })

df = pd.DataFrame(records)
print(f"Successfully matched {len(df)} audio-transcript pairs.")

# 3. Train / Validation split (90% Train, 10% Validation)
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Training samples: {len(train_df)} | Validation samples: {len(val_df)}")
display(train_df.head())

Found 2064 extracted audio files.
Successfully matched 1251 audio-transcript pairs.
Training samples: 1125 | Validation samples: 126


,audio_path,transcript
0,/content/openslr_sinhala_audio/sin_3531_701844...,එහි මූලස්ථානය ස්විස්ටර්ලන්තයේ ජිනිවා නුවර පිහි...
1,/content/openslr_sinhala_audio/sin_3688_106519...,මෙයින් රජතුමා අහසින් වැඩි බව දැන ගත්තේ ය
2,/content/openslr_sinhala_audio/sin_7183_054135...,මෙම ගම්බද පලාත ඔහුගේ පුහුණුව අනුව ගත්තහම වැඩක්...
3,/content/openslr_sinhala_audio/sin_9228_932831...,මෙම ජීවීන් පැමිණෙන යානා පිළිබඳ යම් පැහැදිලි කි...
4,/content/openslr_sinhala_audio/sin_7183_838588...,මෙසේ නොකලොත් අනාගතයේදී මේ යෝග හමුදාව නඩත්තු කල...


In [13]:
# Step 3: Instantiate Dataset & Data Collator
import torch
import torchaudio
from torch.utils.data import Dataset
from dataclasses import dataclass
from typing import Dict, List, Union
from transformers import Wav2Vec2Processor

class SinhalaAudioDataset(Dataset):
    def __init__(self, metadata_df, processor, max_duration=10.0):
        self.df = metadata_df.reset_index(drop=True)
        self.processor = processor
        self.max_duration = max_duration

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        speech_array, sampling_rate = torchaudio.load(row['audio_path'])

        # Convert stereo to mono
        if speech_array.shape[0] > 1:
            speech_array = torch.mean(speech_array, dim=0, keepdim=True)

        # Resample to 16 kHz
        if sampling_rate != 16000:
            resampler = torchaudio.transforms.Resample(sampling_rate, 16000)
            speech_array = resampler(speech_array)

        speech_array = speech_array.squeeze().numpy()

        # Extract features and encode labels
        inputs = self.processor(
            speech_array,
            sampling_rate=16000,
            text=row['transcript']
        )

        return {
            "input_values": torch.tensor(inputs.input_values[0], dtype=torch.float32),
            "labels": torch.tensor(inputs.labels, dtype=torch.long)
        }

@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt"
        )

        labels_batch = self.processor.pad(
            labels=label_features,
            padding=self.padding,
            return_tensors="pt"
        )

        # Mask padding tokens with -100 for CTC loss computation
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch

# Initialize datasets with the processor from Step 3
train_dataset = SinhalaAudioDataset(train_df, processor)
val_dataset = SinhalaAudioDataset(val_df, processor)
data_collator = DataCollatorCTCWithPadding(processor=processor)

In [14]:
import numpy as np
import evaluate
from transformers import Wav2Vec2ForCTC, TrainingArguments, Trainer

# Load base model (MMS 300M or Wav2Vec2 XLSR)
model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/mms-300m",
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),ctc
    ignore_mismatched_sizes=True
)

# Freeze feature encoder to retain low-level acoustic representations
model.freeze_feature_encoder()

cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)
    return {"cer": cer}

# Hyperparameters & Trainer Configuration

training_args = TrainingArguments(
    output_dir="./sinhala_openslr_mms",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    logging_steps=50,
    num_train_epochs=50,
    fp16=True,
    learning_rate=1e-4,
    warmup_steps=300,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,
    report_to="none"
)
#  Execution
trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

trainer.train()

SyntaxError: invalid syntax. Perhaps you forgot a comma? (354674601.py, line 10)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Extract log history from the Trainer instance
log_history = trainer.state.log_history

# Parse log metrics into structured lists
train_steps, train_loss = [], []
eval_steps, eval_loss, eval_cer = [], [], []

for log in log_history:
    if "loss" in log:
        train_steps.append(log["step"])
        train_loss.append(log["loss"])
    if "eval_loss" in log:
        eval_steps.append(log["step"])
        eval_loss.append(log["eval_loss"])
        eval_cer.append(log["eval_cer"])

# Plot Training & Validation Loss
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(train_steps, train_loss, label="Training Loss", color="#1f77b4", linewidth=2)
plt.plot(eval_steps, eval_loss, label="Validation Loss", color="#ff7f0e", linewidth=2, linestyle="--")
plt.xlabel("Steps")
plt.ylabel("Loss")
plt.title("Training vs. Validation Loss")
plt.legend()
plt.grid(True, alpha=0.3)

# Plot Character Error Rate (CER)
plt.subplot(1, 2, 2)
plt.plot(eval_steps, eval_cer, label="Validation CER", color="#2ca02c", linewidth=2, marker="o", markersize=4)
plt.xlabel("Steps")
plt.ylabel("CER")
plt.title("Validation Character Error Rate (CER)")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 1. Install required onnxscript dependency
!pip install -q onnxscript

import torch
from transformers import Wav2Vec2Processor

processor = Wav2Vec2Processor.from_pretrained("./sinhala_processor")
model.eval()

# 2. Create dummy input (1 second of 16kHz audio)
dummy_input = torch.randn(1, 16000, dtype=torch.float32)
onnx_output_path = "sinhala_mms_ctc.onnx"

# 3. Export to ONNX using legacy exporter backend
torch.onnx.export(
    model,
    (dummy_input,),
    onnx_output_path,
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=["input_values"],
    output_names=["logits"],
    dynamic_axes={
        "input_values": {0: "batch_size", 1: "sequence_length"},
        "logits": {0: "batch_size", 1: "sequence_length"},
    },
    dynamo=False,
)

print(f"Model successfully exported to {onnx_output_path}")